In [15]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [16]:
base_dir = Path.cwd()
OUTPUT_DIR = base_dir / "output"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-gex")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.gex.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

RANDOM_SEED = 42
N_JOBS = 4
N_SPLITS = 5

from helpers import run_cv, analyze_feature_importances

In [17]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train = X_train.astype(np.float32)

y_train = pd.read_csv(y_train_path, index_col=0)

X_train, y_train

(             ENSG00000000003  ENSG00000000005  ENSG00000000419  \
 id                                                               
 SJHBBBM2BBR         1.460439        -1.093703         4.276859   
 SJHBBBM2GV5        -1.051043        -1.093703         4.210066   
 SJHBBBM39BT        -1.023765        -1.093703         4.300132   
 SJHBBBM4JQZ        -0.960529        -1.093703         4.433422   
 SJHBBBM4X52        -0.782148        -1.093703         4.030044   
 ...                      ...              ...              ...   
 16-367             -1.093703        -1.093703         3.323508   
 16-372             -0.528834        -1.093703         2.714271   
 16-373              3.905666        -1.093703         3.300383   
 16-374             -0.361243        -1.093703         3.404261   
 16-376              1.012256        -1.093703         3.564771   
 
              ENSG00000000457  ENSG00000000460  ENSG00000000938  \
 id                                                         

In [18]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
B-ALL, high hyperdiploid    389
AML, KMT2A-r                360
B-ALL, ETV6::RUNX1          323
B-ALL, BCR::ABL1-like       267
AML, RUNX1::RUNX1T1         253
AML, NPM1alt                158
B-ALL, KMT2A-r              153
B-ALL, PAX5alt              153
T-ALL                       125
B-ALL, DUX4-r               123
AML, CBFB::MYH11            118
B-ALL, BCR::ABL1             97
B-ALL, TCF3::PBX1            88
B-ALL, hypodiploid           87
AML, CEBPAalt                79
AML, NUP98-r                 68
B-ALL, ZNF384-r              59
B-ALL, ETV6::RUNX1-like      48
B-ALL, MEF2D-r               40
B-ALL, iAMP21                36
AML, DEK::NUP214             31
B-ALL, PAX5 P80R             27
AML, CBFA2T3::GLIS2          25
Non-leukemic                 15
AML, APL-PML::RARA           14
B-ALL, NUTM1-r               13
B-ALL, MYC-r                 12
AML, ETV6-r                   8
AML, KAT6A-r                  8
B-ALL, HLF-r       

In [19]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_train

array(['B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', ..., 'T-ALL', 'B-ALL, high hyperdiploid',
       'B-ALL, BCR::ABL1-like'], shape=(3193,), dtype=object)

In [20]:
cv_results = run_cv(X_train, y_train, n_jobs=10)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time=32.4min
[CV] END .................................................... total time=32.9min
[CV] END .................................................... total time=34.0min
[CV] END .................................................... total time=35.1min
[CV] END .................................................... total time=35.2min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.868 ± 0.017
Gap:            0.132


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed: 35.2min finished


In [21]:
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR,cv_results, X_train, suffix="1", freq_threshold=1)
X_reduced = X_train[selected_features]
X_reduced

Selected 5487 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/gex/output/selected_features.1.txt.


,ENSG00000184507,ENSG00000079102,ENSG00000248329,ENSG00000124302,ENSG00000159217,ENSG00000164853,ENSG00000133107,ENSG00000157680,ENSG00000180190,ENSG00000185046,...,ENSG00000136689,ENSG00000183323,ENSG00000134717,ENSG00000213859,ENSG00000197694,ENSG00000172053,ENSG00000149926,ENSG00000205670,ENSG00000119328,ENSG00000040199
id,,,,,,,,,,,,,,,,,,,,,
SJHBBBM2BBR,-0.948134,-1.093703,-1.093703,-1.093703,-1.093703,-1.093703,-1.093703,-1.093703,-0.918452,0.201662,...,6.264262,4.444773,6.115304,2.817279,6.699731,-1.093703,1.004276,0.270711,4.389894,3.621396
SJHBBBM2GV5,-0.988235,-1.093703,-0.594273,-1.093703,8.169499,-1.093703,-0.683797,-1.079131,-0.050286,1.871542,...,0.853071,4.860859,5.557657,3.876947,9.029334,-1.093703,-0.347495,0.468178,4.716640,3.414933
SJHBBBM39BT,-1.093703,-1.093703,-0.987930,-1.093703,8.396330,-1.093703,2.802036,-0.955134,-0.339056,-0.061098,...,1.387706,4.295958,6.579481,3.625267,7.086385,-1.093703,0.717433,-0.295776,5.291368,3.057661
SJHBBBM4JQZ,-1.093703,-1.093703,-1.093703,-1.093703,7.624845,-1.093703,1.082320,-0.835160,1.393891,0.808224,...,0.888958,4.306052,6.349683,3.412633,7.561121,-1.093703,0.528642,0.022892,5.059034,3.253351
SJHBBBM4X52,-1.093703,-1.051298,-0.895005,-1.093703,8.010935,-1.093703,0.997237,-1.024795,1.185382,0.181840,...,3.288370,4.896974,5.918674,3.215110,7.944075,-1.093703,0.105029,-0.087887,4.665517,3.958042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16-367,-1.093703,0.428460,-1.093703,-1.093703,-1.067074,-1.093703,-1.093703,-1.060340,-0.694303,-1.047762,...,4.797078,3.821796,6.143647,4.391636,8.217925,8.329537,0.512373,-0.335850,5.087320,2.959791
16-372,-1.045965,-0.936247,-1.093703,-0.907553,-1.052900,-1.093703,-1.093703,-1.042643,-0.685058,-0.622626,...,5.302874,4.047936,6.060882,3.532571,7.888659,8.928311,1.100732,-0.282475,5.072904,3.497490
16-373,-1.093703,-1.093703,-1.093703,-1.093703,-1.093703,-1.093703,0.442331,-1.004391,-0.501166,-1.062193,...,3.449622,3.758221,6.197735,5.005531,8.470355,8.445149,2.827875,-0.345326,4.947685,3.882189


In [22]:
cv_results = run_cv(X_reduced, y_train, n_jobs=10)
cv_results

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time= 9.4min
[CV] END .................................................... total time= 9.6min
[CV] END .................................................... total time=10.4min
[CV] END .................................................... total time=10.8min
[CV] END .................................................... total time=11.2min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.859 ± 0.020
Gap:            0.141


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed: 11.2min finished


{'fit_time': array([573.32600927, 670.60132217, 625.96105027, 561.83592296,
        649.41791177]),
 'score_time': array([0.03168464, 0.02426815, 0.03051519, 0.03372693, 0.02872467]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.85047749, 0.88852881, 0.87625623, 0.84841635, 0.83222906]),
 'train_score': array([1., 1., 1., 1.,

In [23]:
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR,cv_results, X_reduced, suffix="2", freq_threshold=1)
X_reduced_2 = X_reduced[selected_features]
X_reduced_2

Selected 1121 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/gex/output/selected_features.2.txt.


,ENSG00000184507,ENSG00000079102,ENSG00000248329,ENSG00000101115,ENSG00000172752,ENSG00000097007,ENSG00000132185,ENSG00000167880,ENSG00000177469,ENSG00000188064,...,ENSG00000170396,ENSG00000130702,ENSG00000187325,ENSG00000169213,ENSG00000186980,ENSG00000182747,ENSG00000144681,ENSG00000072954,ENSG00000181007,ENSG00000107819
id,,,,,,,,,,,,,,,,,,,,,
SJHBBBM2BBR,-0.948134,-1.093703,-1.093703,0.900417,-1.071515,6.815446,1.969651,5.079339,2.107005,-1.011734,...,-0.874603,2.447312,7.192753,-0.881324,-1.093703,-0.921531,-1.093703,2.686805,3.305760,4.890347
SJHBBBM2GV5,-0.988235,-1.093703,-0.594273,6.403160,-1.069940,6.763022,2.974250,4.091602,3.714177,-1.049207,...,-0.386459,0.688034,4.789256,-0.366426,-1.093703,-1.093703,-0.286578,1.501182,3.205284,4.291543
SJHBBBM39BT,-1.093703,-1.093703,-0.987930,0.122253,-0.979386,6.635201,3.343168,1.923664,2.778868,-1.093703,...,-0.486244,2.982402,5.662900,1.125583,-1.093703,-1.093703,-0.932555,3.153598,3.929690,4.207279
SJHBBBM4JQZ,-1.093703,-1.093703,-1.093703,2.117265,-1.055608,6.177442,4.900558,0.128974,3.459446,-1.093703,...,-0.554632,1.081338,5.842680,-0.789552,-1.093703,-1.093703,-0.794956,3.949895,3.154709,5.872823
SJHBBBM4X52,-1.093703,-1.051298,-0.895005,6.255462,-1.018710,6.255557,3.568772,2.859744,3.909604,-1.093703,...,-0.939348,1.209302,4.668585,-0.578701,-1.093703,-0.944323,-0.154415,3.388016,3.542804,4.657595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16-367,-1.093703,0.428460,-1.093703,-0.595068,-1.066356,7.519018,1.647460,3.696073,6.558373,-0.945385,...,0.295889,1.043754,5.757451,-0.340621,-1.093703,-0.392349,1.844438,1.634769,3.073470,5.784753
16-372,-1.045965,-0.936247,-1.093703,-0.991571,-0.721107,8.540172,3.086241,6.243773,1.779481,-0.804093,...,-0.509767,2.464188,5.937675,-1.078249,-1.093703,-1.093703,-0.628955,1.487672,3.129442,6.358287
16-373,-1.093703,-1.093703,-1.093703,-0.297175,-1.002446,8.535685,1.864524,6.393374,8.678990,-0.864220,...,-0.458841,1.755817,6.775340,0.823834,-1.093703,-0.879317,0.747372,1.939339,3.053915,6.525223


In [24]:
cv_results = run_cv(X_reduced_2, y_train, n_jobs=10)
cv_results

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time= 1.8min
[CV] END .................................................... total time= 1.9min
[CV] END .................................................... total time= 1.9min
[CV] END .................................................... total time= 2.2min
[CV] END .................................................... total time= 2.3min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.887 ± 0.013
Gap:            0.113


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed:  2.3min finished


{'fit_time': array([110.20223927, 135.19214797, 134.52812672, 115.83466554,
        115.76836014]),
 'score_time': array([0.01914787, 0.01436567, 0.01656961, 0.01893449, 0.01847935]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.87570037, 0.88970688, 0.87797436, 0.88185673, 0.91149566]),
 'train_score': array([1., 1., 1., 1.,

In [25]:
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR,cv_results, X_reduced_2, suffix="3", freq_threshold=1)

Selected 196 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/gex/output/selected_features.3.txt.


In [26]:
X_reduced_3 = X_reduced_2[selected_features]
X_reduced_3

,ENSG00000184507,ENSG00000079102,ENSG00000248329,ENSG00000101115,ENSG00000145794,ENSG00000105642,ENSG00000108511,ENSG00000168004,ENSG00000185046,ENSG00000164100,...,ENSG00000161031,ENSG00000101224,ENSG00000132478,ENSG00000076344,ENSG00000006453,ENSG00000047648,ENSG00000139910,ENSG00000070601,ENSG00000125869,ENSG00000105737
id,,,,,,,,,,,,,,,,,,,,,
SJHBBBM2BBR,-0.948134,-1.093703,-1.093703,0.900417,-0.116294,0.996957,-1.029480,-0.969128,0.201662,-1.093703,...,-0.879186,6.211644,6.470520,-1.007059,2.633660,0.516906,-1.093703,2.804746,6.079550,5.348482
SJHBBBM2GV5,-0.988235,-1.093703,-0.594273,6.403160,0.075866,8.961658,-1.024992,-0.572768,1.871542,-0.906562,...,-0.813060,6.527030,6.507983,-0.831893,2.176762,0.395408,4.217975,-0.981697,4.305465,5.381610
SJHBBBM39BT,-1.093703,-1.093703,-0.987930,0.122253,-0.256646,8.385358,-0.981690,-0.879641,-0.061098,-1.093703,...,1.070762,6.274668,6.134863,-1.016612,2.175858,0.332830,3.904323,-1.030837,4.139028,4.949487
SJHBBBM4JQZ,-1.093703,-1.093703,-1.093703,2.117265,0.442155,8.352180,-0.883089,-0.985546,0.808224,-1.093703,...,0.159603,5.233226,5.809965,-1.018607,-0.020343,0.355009,4.035835,-0.917295,2.780713,4.881221
SJHBBBM4X52,-1.093703,-1.051298,-0.895005,6.255462,-0.142965,8.341070,-0.984867,-0.203198,0.181840,-1.093703,...,-0.822317,5.887336,5.971149,-0.947640,1.813479,-1.093703,4.441238,-1.032649,5.695688,4.817143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16-367,-1.093703,0.428460,-1.093703,-0.595068,-0.851867,-0.747873,-1.014810,3.889202,-1.047762,8.056761,...,-0.612513,7.821895,6.911818,-0.751723,1.509739,0.520639,3.300325,-0.965379,0.167092,2.279875
16-372,-1.045965,-0.936247,-1.093703,-0.991571,2.738100,-0.750976,-1.093703,-0.622965,-0.622626,-1.064456,...,-0.709859,7.327261,7.525078,-0.508573,0.571597,0.444771,-1.093703,-0.993907,7.677779,7.645868
16-373,-1.093703,-1.093703,-1.093703,-0.297175,-0.741214,-0.507046,-1.039399,3.522191,-1.062193,8.040997,...,-0.184458,7.642885,7.310933,-1.056562,0.064627,-0.005836,2.524428,-0.501201,1.704440,1.939602


In [ ]:
cv_results = run_cv(X_reduced_3, y_train, n_jobs=10)
cv_results

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time=  15.3s
[CV] END .................................................... total time=  16.7s
[CV] END .................................................... total time=  18.8s
[CV] END .................................................... total time=  18.6s
[CV] END .................................................... total time=  19.6s
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.883 ± 0.011
Gap:            0.117


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed:   20.6s finished


{'fit_time': array([19.57551718, 18.7833097 , 18.58633947, 15.23714042, 16.65631557]),
 'score_time': array([0.01309896, 0.01545811, 0.01509404, 0.01549149, 0.01755834]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.89988887, 0.87317761, 0.87872865, 0.87194609, 0.8898617 ]),
 'train_score': array([1., 1., 1., 1., 1.])}

In [28]:
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR,cv_results, X_reduced_3, suffix="4", freq_threshold=1)

Selected 34 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/gex/output/selected_features.4.txt.


In [29]:
X_reduced_4 = X_reduced_3[selected_features]
X_reduced_4
cv_results = run_cv(X_reduced_4, y_train, n_jobs=10)
cv_results

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[CV] END .................................................... total time=   2.2s
[CV] END .................................................... total time=   2.4s
[CV] END .................................................... total time=   2.5s
[CV] END .................................................... total time=   2.8s
[CV] END .................................................... total time=   2.9s
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.769 ± 0.034
Gap:            0.231


[Parallel(n_jobs=10)]: Done   5 out of   5 | elapsed:    4.2s finished


{'fit_time': array([2.8836832 , 2.77803326, 2.42452574, 2.15480733, 2.49279833]),
 'score_time': array([0.01109934, 0.01219511, 0.01267052, 0.01338983, 0.01259184]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.80216669, 0.73702136, 0.78452646, 0.80009507, 0.71870573]),
 'train_score': array([1., 1., 1., 1., 1.])}